In [10]:
import pandas as pd
import numpy as np
import re
import os

In [22]:
bench_name="flask"

In [24]:
eval_folder_path = os.path.dirname(os.path.dirname(os.getcwd()))

#Prediction path
path=eval_folder_path+f"/evaluation_results/{bench_name}/"
l=os.listdir(path)
l.sort()

#Labels
gold=pd.read_json(eval_folder_path+f"/benchmarks/{bench_name}/{bench_name}_en.jsonl",lines=True)

print("Files to evaluate: ")
display(l)

Files to evaluate: 


[]

In [25]:
display(gold["criteria"].value_counts())

criteria
Comprehension                468
Completeness                 215
Commonsense Understanding    198
Factuality                   191
Readability                  180
Logical Correctness          180
Insightfulness               159
Conciseness                  147
Logical Robustness           104
Logical Efficiency            75
Metacognition                 44
Harmlessness                  40
Name: count, dtype: int64

## 1. Metrics

In [ ]:
def extract_value(text, lax=True, print_no_match=False):
    """Extracts the score. If lax=True, it will also consider outputs that do not match the prompted format but output a score."""

    pattern = r"\[(RESULT|EMAITZA|RESULTADO)\]\s*([1-5])"
    match = re.search(pattern, text)

    test=re.search(r"\[FORMAT ERROR\]\s*([1-5])", text)
    if match:
        return int(match.group(2).strip())
    elif test and lax:
        return int(test.group(1).strip())
    if print_no_match:
        print(text,"\n\n")
    return 0

In [ ]:
from sklearn.metrics import mean_squared_error
from scipy import stats
import krippendorff

def exact_match(pred,label):
    return (np.mean([1 if x==y else 0 for x,y in zip(pred,label)]))

def get_correlations(l,gold,print_results=False,mean_dict=False):

    result_dict={}
    for i,doc in enumerate(l):
        print(f"Processing document {i+1}/{len(l)}",end="\r")
        train_lang=doc.split("-")[0]
        aux=doc.split(f"_{bench_name}_")
        test_lang=aux[-1][:-6]
        model_name=aux[0][len(train_lang)+1:]

        if print_results:
            print(doc)
        file_result_dict={}

        d=pd.read_json(path+doc,lines=True).loc[gold.index]
        
        try:
            d["scores"]=d.apply(lambda x: [extract_value(x["outputs_0"]),extract_value(x["outputs_1"]),extract_value(x["outputs_2"])], axis=1,)
        except:
            print("Single prediction in", doc)
            d["scores"]=d.apply(lambda x: extract_value(x["outputs_0"]), axis=1)

        label = gold["gold"].to_list()
        pred=d.apply(lambda x:stats.mode(x["scores"], keepdims=False).mode, axis=1,).to_list()

        #-------------Calculate metrics-------------
        #Pearson correlation
        pearson=stats.pearsonr(pred,label)
        pears=pearson.statistic
        p_val=pearson.pvalue

        #Exact match
        em=exact_match(pred,label)
        #Mean Squared Error
        mse=mean_squared_error(label,pred)
        #Krippendorff's alpha. Only considered if there are multiple predictions per item, otherwise nan.
        try:
            k_alpha=krippendorff.alpha(reliability_data=np.array(d["scores"].to_list()).T, level_of_measurement='nominal')
        except:
            k_alpha=np.nan
        #Missing predictions
        missing=pred.count(0)

        #---------------Save results----------------
        file_result_dict["model"]=model_name
        file_result_dict["train_lang"]=train_lang
        file_result_dict["pearson"]=pears
        file_result_dict["p_val"]=p_val
        file_result_dict["exact_match"]=em
        file_result_dict["mse"]=mse
        if not np.isnan(k_alpha):
            file_result_dict["krippendorff_alpha"]=k_alpha
        file_result_dict["missing_predictions"]=missing

        try:
            result_dict[test_lang].append(file_result_dict)
        except:
            result_dict[test_lang]=[file_result_dict]

        #---------------Print results----------------
        if print_results:
            print("Correlations")
            print("Pearson",round(pears,3), "   (p-value of", round(p_val,3),")")
            print("Exact Match",round(em,3))
            print("MSE",round(mse,3))
            if not np.isnan(k_alpha):
                print("krippendorffs alpha",round(k_alpha,3))
            print(f"Missing predictions (count of 0 s): {missing}")
            print("\n")
    if print_results:
        print("___________________________________________________________________________________________________")

    if mean_dict:
        import numbers

        keys = list(result_dict.keys())
        n = len(result_dict[keys[0]])  # number of epochs (3)

        mean_list = []

        for i in range(n):
            dicts = [result_dict[k][i] for k in keys]
            out = {}
            
            for field in dicts[0]:
                vals = [d[field] for d in dicts]
                
                if isinstance(vals[0], numbers.Number):
                    out[field] = sum(vals) / len(vals)
                else:
                    out[field] = vals[0]  # keep string
            
            mean_list.append(out)

        return(mean_list)
    else:
        return result_dict

In [ ]:
for cr in gold["criteria"].unique():
    print("Criterion:",cr)
    get_correlations(l,gold[gold["criteria"]==cr],print_results=True)

## 2. Plot of the effects of finetuning

In [ ]:
l_Llama=[x for x in os.listdir(path) if "Llama-3.1-70B-Instruct" in x and "Latxa" not in x]
l_Latxa=[x for x in os.listdir(path) if "Latxa-Llama-3.1-70B-Instruct" in x]

In [ ]:
import matplotlib.pyplot as plt

all_gold = gold.copy()

models = [
    "Latxa-Llama-3.1-70B-Instruct",
    "Latxa-Llama-3.1-70B-Instruct-ep1",
    "Latxa-Llama-3.1-70B-Instruct-ep2",
    "Latxa-Llama-3.1-70B-Instruct-ep3",
]

colors = ["#5f99c3", "#e8e066", "#EDAA2D", "#c85b3f"]

groups = {
    "Logical Thinking": [
        "Logical Correctness",
        "Logical Robustness",
        "Logical Efficiency",
    ],
    "Background Knowledge": [
        "Factuality",
        "Commonsense Understanding",
    ],
    "Problem Handling": [
        "Comprehension",
        "Insightfulness",
        "Completeness",
        "Metacognition",
    ],
    "User Alignment": [
        "Conciseness",
        "Readability",
        "Harmlessness",
    ],
}

display_labels = {
    "Commonsense Understanding": "Commonsense\nUnderstanding"
}

ordered_criteria = [c for g in groups.values() for c in g]
criteria = all_gold["criteria"].unique()

rows = []
for cr in ordered_criteria:
    d = get_correlations( l_Latxa, all_gold[all_gold["criteria"] == cr], "human", mean_dict=True)
    aux = pd.DataFrame(d)
    aux["criteria"] = cr
    rows.append(aux)

df = pd.concat(rows, ignore_index=True)
df = df[df["model"].isin(models)]
pivot = df.pivot(index="criteria", columns="model", values="pearson")
pivot = pivot.reindex(ordered_criteria)

N = len(pivot.index)
bar_width = 0.2
x = np.arange(N)

fig, ax = plt.subplots(figsize=(13, 5))

for i, model in enumerate(models):
    if model in pivot.columns:
        ax.bar(
            x + i * bar_width,
            pivot[model].values,
            width=bar_width,
            color=colors[i],
            label=i
        )

tick_labels = [display_labels.get(c, c) for c in pivot.index]
ax.set_xticks(x + bar_width * (len(models) - 1) / 2)
ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)

start = 0
group_annotations = []

for g, crits in groups.items():
    end = start + len(crits)
    x_left  = start - bar_width / 2
    x_right = end - 1 + bar_width * (len(models) - 1) + bar_width / 2
    x_center = (x_left + x_right) / 2
    group_annotations.append((g, x_left, x_right, x_center))

    if end < N:
        line_x = end - bar_width / 2
        ax.axvline(line_x, color="gray", linestyle="--", alpha=0.5)

    start = end

#ax.set_ylabel("Pearson Correlation")
ax.set_title("Pearson Correlations per Criterion", pad=52)
ax.legend(title="Epoch", loc="upper right")

for g, x_left, x_right, x_center in group_annotations:
    # bracket line
    ax.annotate(
        "",
        xy=(x_right, 1.13), xycoords=("data", "axes fraction"),
        xytext=(x_left, 1.13), textcoords=("data", "axes fraction"),
        arrowprops=dict(arrowstyle="-", color="gray", lw=1.2)
    )
    # group label
    ax.text(
        x_center, 1.16, g,
        ha="center", va="bottom",
        transform=ax.get_xaxis_transform(),
        fontsize=9.5, fontweight="bold", color="#333333"
    )

fig.subplots_adjust(bottom=0.25, top=0.82)
plt.show()